In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime
from scipy import stats
from sklearn.model_selection import TimeSeriesSplit

def load_and_preprocess_data(file_path):
    """加载并预处理指数数据"""
    try:
        df = pd.read_csv(file_path)
        
        # 将日期时间转换为datetime类型
        df['datetime'] = pd.to_datetime(df['date'] + ' ' + df['time'])
        
        # M1提取日期
        df['date_only'] = ___________________
        # M2提取小时
        df['hour'] = ___________________
        
        # 处理异常值
        # pct_change字段异常处理
        df['pct_change'] = pd.to_numeric(df['pct_change'], errors='coerce').fillna(0)
        
        return df
        
    except FileNotFoundError:
        print("错误：找不到CSV文件，请确保数据文件在正确路径下。")
        return None
    except Exception as e:
        print(f"发生未知错误：{e}")
        return None

def feature_engineering_1(df):
    """特征1：指数在不同时间段的平均涨跌幅"""
    print("\n开始构建特征1：指数在不同时间段的平均涨跌幅")
    
    # 将一天划分为早盘/午盘/收盘前
    def time_period(hour):
        if 9 <= hour < 11:
            return 'morning'
        elif 11 <= hour < 14:
            return 'noon'
        else:
            return 'afternoon'
    
    df['time_period'] = df['hour'].apply(time_period)
    
    # M3计算指数在每个时间段的平均涨跌幅，可多行代码段实现
    period_index_change = ___________________
    _________________________________________
    _________________________________________
    print(f"计算了 {len(period_index_change)} 个指数×日期的平均涨跌幅")
    print(f"平均涨跌幅: {period_index_change['avg_pct_change'].mean():.4f}")
    
    return df, period_index_change

def feature_engineering_2(df):
    """特征2：板块 × 市场阶段的历史涨跌概率"""
    print("\n开始构建特征2：板块 × 市场阶段的历史涨跌概率")
    
    # 计算涨跌概率
    df['up'] = (df['pct_change'] > 0).astype(int)
    # M4 计算不同板块的指数在不同市场阶段的平均涨跌幅，可多行代码段实现
    sector_phase_probs = ___________________    
    _________________________________________
    _________________________________________
    print(f"计算了 {len(sector_phase_probs)} 个板块×市场阶段的历史涨跌概率")
    print(f"平均涨跌概率: {sector_phase_probs['up_probability'].mean():.4f}")
    
    return df, sector_phase_probs

def feature_engineering_3(df):
    """特征3：指数 × 板块的短期涨跌趋势偏好"""
    print("\n开始构建特征3：指数 × 板块的短期涨跌趋势偏好")
    
    # 使用过去5个交易日的涨跌趋势
    df['trend_5'] = df.groupby(['index_id'])['pct_change'].rolling(window=5).sum().reset_index(0,drop=True)
     # M5 计算不同板块的指数在过去5个交易日的涨跌趋势，可多行代码段实现
    index_sector_trend = ___________________   
    _________________________________________
    _________________________________________
    print(f"计算了 {len(index_sector_trend)} 个指数×板块的短期涨跌趋势")
    print(f"平均趋势值: {index_sector_trend['avg_trend_5'].mean():.4f}")
    
    return df, index_sector_trend

def build_golden_test_set(df):
    """构建黄金测试集并进行偏差检测"""
    print("\n开始构建黄金测试集并进行偏差检测...")
    
    df_sorted = df.sort_values('datetime')
    
    # M6 使用TimeSeriesSplit划分训练集和测试集，可多行代码段实现
    _________________________________________
    _________________________________________
    _________________________________________
    
    print(f"训练集样本数: {len(train_data)}, 测试集样本数: {len(test_data)}")
    
    # 偏差分析：不同板块涨跌概率分布
    print("\n偏差分析：不同板块涨跌概率")
    overall_probs = df.groupby('sector')['up'].mean()
    test_probs = test_data.groupby('sector')['up'].mean()
    
    all_sectors = sorted(set(df['sector'].unique()).union(set(test_data['sector'].unique())))
    overall_probs = overall_probs.reindex(all_sectors, fill_value=0)
    test_probs = test_probs.reindex(all_sectors, fill_value=0)
    
    # M7 KS检验
    _________________________________________
    print(f"KS检验结果: statistic={ks_stat:.4f}, p-value={ks_p:.4f}")
    
    return train_data, test_data

def main():
    """主函数，执行完整的数据处理流程"""
    
    df = load_and_preprocess_data('2.1.1.csv')
    if df is None:
        return
    
    # 特征工程
    df, period_index_change = feature_engineering_1(df)
    df, sector_phase_probs = feature_engineering_2(df)
    df, index_sector_trend = feature_engineering_3(df)
    
    # 保存高质量训练集
    high_quality_train = df[['index_id','date_only','time_period','pct_change','sector','market_phase']]
    high_quality_train.to_csv('high_quality_train_index.csv', index=False)
    print("\n高质量训练集已保存为 high_quality_train_index.csv")
    
    # 构建黄金测试集
    train_data, test_data = build_golden_test_set(df)
    
    print("\n所有任务已完成！")

if __name__ == "__main__":
    main()
